# Week 10, Part 2: SQLite & SQL from Python
**Phase 2: Python for Data Analysis**

You write SQL every day, but always against someone else's database. This notebook gives you **your own**: a local analytics database you design, load, and query, from Python, with no server and no account.

**Why a database when you already have Pandas?**

| | DataFrame / CSV | SQLite database |
|---|---|---|
| Survives closing Python | Only if you save/reload | **Yes**, it's a file |
| Rules (no duplicates, no negative prices) | None, bad data flows through | **Enforced** by constraints |
| Re-run a load safely | Duplicates or overwrites | **Upsert**: idempotent |
| Query part of a big dataset | Load it all into RAM | Indexed lookups, loads only the result |
| Multiple tools reading it | Passing files around | One file, anyone with SQL |
| Heavy reshaping, plotting, ML | **Great** | Awkward |

They're complements: **SQLite stores and filters; Pandas reshapes and analyses.** Pulling a SQL result into a DataFrame is one line (section 11).

**What is SQLite?** A complete SQL database engine in a library that ships **inside Python** (`import sqlite3`), storing everything in one ordinary file. No install, no server, no password. It's the most widely deployed database in the world (every phone has many), and a perfect place to learn production habits.

**Prerequisites:** `lesson.ipynb` (this week), Week 8 (Pandas), and your SQL skills, which mostly transfer. The differences from DuneSQL (Trino) are in section 12.

## 0. Setup

In [ ]:
import os
import sqlite3
import sys
import tempfile
import time

import numpy as np
import pandas as pd

sys.path.insert(0, "../../phase-1-python-fundamentals/week-05-functions-modules")
sys.path.append("../week-08-pandas-numpy")
from blockchain_utils import fmt_usd, fmt_large

from db_tools import connect, init_db, table_counts, SCHEMA
from fixtures import FixtureSession
from pipeline import run_pipeline
from sample_data import make_dex_trades

print("sqlite library version:", sqlite3.sqlite_version)       # window functions need >= 3.25, upsert >= 3.24

## 1. Connection, cursor, and your first table

Three objects:

- **Connection**: the open database (a file, or `":memory:"` for a throwaway one in RAM)
- **Cursor**: runs statements and walks results (`conn.execute()` creates one for you)
- **Row**: one result row. By default a plain tuple; set `row_factory = sqlite3.Row` and you can read columns **by name**, like a dict.

```
connect(path) -> Connection -> execute(sql, params) -> Cursor -> fetchone() / fetchall() / iterate
```

In [ ]:
db = sqlite3.connect(":memory:")            # RAM-only scratch database; vanishes when closed
db.row_factory = sqlite3.Row                # rows readable by column name

db.execute('''CREATE TABLE pools (
    pool    TEXT PRIMARY KEY,
    chain   TEXT NOT NULL,
    tvl_usd REAL
)''')

db.execute("INSERT INTO pools VALUES ('WETH-USDC', 'Ethereum', 210000000)")
db.execute("INSERT INTO pools VALUES ('WBTC-WETH', 'Ethereum', 98000000)")
db.commit()                                 # <- without this the inserts stay in an OPEN transaction (see the note below)

rows = db.execute("SELECT * FROM pools ORDER BY tvl_usd DESC").fetchall()
print(type(rows[0]).__name__, "| columns:", rows[0].keys())
for r in rows:
    print(r["pool"], fmt_large(r["tvl_usd"]))     # by name, not index

Three ways to read results, from least to most memory:

| Method | Returns | Use when |
|---|---|---|
| `cursor.fetchone()` | one row (or `None`) | you expect exactly one row |
| `cursor.fetchall()` | list of all rows | small result |
| `for row in cursor:` | one row at a time (a generator!) | big result, same laziness idea as `lesson.ipynb` |

### 1.1 The commit gotcha (read this once, save yourself hours)

Python's `sqlite3` **silently opens a transaction at your first `INSERT`/`UPDATE`/`DELETE`** and leaves it open until you call `commit()` (or use `with conn:`). Until then your changes are **invisible to other connections, and lost if the program exits or an error triggers a rollback.** A forgotten `commit()` is the classic "my data vanished" bug.

(`SELECT`s don't open one, which is why reading works fine and the bug stays hidden until later.)

## 2. Transactions: all or nothing

A **transaction** groups statements so they succeed **together or not at all**. Without one, a crash halfway through a load leaves your database half-updated, which is the worst kind of corruption because nothing *looks* broken.

In Python's `sqlite3`, using the connection as a context manager gives you a transaction: **commit on success, rollback on exception**.

In [ ]:
db.execute("CREATE TABLE ledger (wallet TEXT, amount REAL)")

try:
    with db:                                              # begin transaction
        db.execute("INSERT INTO ledger VALUES ('0xaaa', 100)")
        db.execute("INSERT INTO ledger VALUES ('0xbbb', 250)")
        raise RuntimeError("API died halfway through the load")
except RuntimeError as e:
    print("caught:", e)

print("rows after failed load:", db.execute("SELECT COUNT(*) FROM ledger").fetchone()[0], " <- rolled back, no partial data")

with db:                                                  # this one succeeds -> committed
    db.executemany("INSERT INTO ledger VALUES (?, ?)", [("0xaaa", 100), ("0xbbb", 250)])
print("rows after good load  :", db.execute("SELECT COUNT(*) FROM ledger").fetchone()[0])

**Remember:** `with conn:` does **not** close the connection; it only manages the transaction. Close explicitly with `conn.close()` when finished.

## 3. SQLite's type system (and the wei trap, again)

SQLite is **dynamically typed**, which will surprise you coming from Trino/Postgres. A column's declared type is only a *preference* (an **affinity**); SQLite stores whatever you give it and converts when it sensibly can.

| Declared | Affinity | Stores |
|---|---|---|
| `INTEGER` | integer | 64-bit signed int (±9.22 × 10¹⁸) |
| `REAL` | float | 8-byte float |
| `TEXT` | text | strings |
| `BLOB` | none | raw bytes |
| `NUMERIC` | numeric | int or float, whichever fits |

Consequences for blockchain data:

- **There is no date/time type.** Store dates as `'YYYY-MM-DD'` TEXT (sorts and compares correctly) and timestamps as INTEGER unix seconds, then use `date()`/`strftime()` to work with them.
- **There is no boolean.** Use `0`/`1` (add a `CHECK (col IN (0,1))`).
- **There is no `uint256`/`DECIMAL(38,0)`.** Integers stop at int64, which is **about 9.2 ETH in wei.**

That last one is the same trap as Week 8's NumPy overflow, but SQLite has *four different failure modes*. I tested each so you don't have to learn them in production:

In [ ]:
t = sqlite3.connect(":memory:")
big = 12_500_000_000_000_000_000                # 12.5 ETH in wei: larger than int64's max of 9,223,372,036,854,775,807
t.execute("CREATE TABLE as_int (v INTEGER)")
t.execute("CREATE TABLE as_txt (v TEXT)")

# 1) A Python int too big for SQLite: LOUD failure (good)
try:
    t.execute("INSERT INTO as_int VALUES (?)", (big,))
except OverflowError as e:
    print("1) bind huge Python int  ->", type(e).__name__)

# 2) The same number as a STRING into an INTEGER column: SILENT conversion to a lossy float (bad!)
t.execute("INSERT INTO as_int VALUES (?)", (str(big),))
print("2) text into INTEGER col ->", t.execute("SELECT v, typeof(v) FROM as_int").fetchone()[:], " <- became a float")

# 3) TEXT column: exact
t.execute("INSERT INTO as_txt VALUES (?)", (str(big),))
print("3) TEXT column           ->", t.execute("SELECT v, typeof(v) FROM as_txt").fetchone()[:])

# 4) CAST to INTEGER: SILENTLY saturates at the int64 max (wrong answer, no error)
print("4) CAST(text AS INTEGER) ->", t.execute("SELECT CAST(v AS INTEGER) FROM as_txt").fetchone()[0])

# 5) SUM of integers that overflow: loud error
t.execute("CREATE TABLE sums (v INTEGER)")
t.executemany("INSERT INTO sums VALUES (?)", [(9 * 10**18,), (9 * 10**18,)])
try:
    t.execute("SELECT SUM(v) FROM sums").fetchone()
except sqlite3.OperationalError as e:
    print("5) SUM overflows        ->", e)

**The rule:** store wei (and any uint256 token amount) as **`TEXT`**, do exact arithmetic in **Python ints**, and convert to float ETH only at the very end. Cases 2 and 4 are the dangerous ones because the database gives you a plausible-looking wrong number.

How wrong? Summing our 250 demo transactions as floats instead of exact ints is off by tens of thousands of wei, small in ETH but a different number, which breaks reconciliation (and Exercise 15 checks for exactly this).

> This is why `db_tools.SCHEMA` declares `value_wei TEXT`. In SQL you can still *filter* on it with `LENGTH(value_wei)` tricks or `CAST(value_wei AS REAL)` for rough comparisons, but never `SUM` it for accounting.

## 4. Parameterised queries (and SQL injection)

Never build SQL by pasting values into a string. Use **placeholders**; the driver sends the SQL and the values **separately**, so a value can never be interpreted as SQL.

Let's see why, using a real table: load Week 8's trades into SQLite.

In [ ]:
trades = make_dex_trades()
trades.to_sql("trades", db, index=False)        # DataFrame -> table in one line (see section 7 for gotchas)
print(db.execute("SELECT COUNT(*) FROM trades").fetchone()[0], "trades loaded")

wallet = trades["wallet"].iloc[0]
user_input = "nobody' OR '1'='1"                # what an attacker (or a bug) might send

# ❌ f-string: the input becomes part of the SQL
sql = f"SELECT COUNT(*) FROM trades WHERE wallet = '{user_input}'"
print("f-string query :", sql)
print("   rows matched:", db.execute(sql).fetchone()[0], " <- matched EVERYTHING")

# ✅ placeholder: the input is just data
print("placeholder    :", db.execute("SELECT COUNT(*) FROM trades WHERE wallet = ?", (user_input,)).fetchone()[0], "rows")
print("real wallet    :", db.execute("SELECT COUNT(*) FROM trades WHERE wallet = ?", (wallet,)).fetchone()[0], "rows")

In a local analytics DB the "attacker" is usually a *bug*: a token symbol containing a quote (`O'Neil`), a pasted address with a stray space. Placeholders make these harmless. On any service that takes outside input, they're the difference between a working app and a breach.

Placeholder styles:

```python
conn.execute("... WHERE a = ? AND b = ?", (1, 2))                    # positional
conn.execute("... WHERE a = :a AND b = :b", {"a": 1, "b": 2})        # named: clearer for many params
```

Two limits to know:

1. **`IN` lists need one `?` per value**: build the placeholder string (safely, from a count), not the values.
2. **Identifiers (table/column names) can't be parameters.** If a table name comes from a variable, check it against an allow-list.

In [ ]:
protocols = ["Uniswap", "Curve"]
placeholders = ",".join("?" * len(protocols))                 # "?,?"  (built from a COUNT, not from user text)
sql = f"SELECT protocol, COUNT(*) FROM trades WHERE protocol IN ({placeholders}) GROUP BY protocol"
print(db.execute(sql, protocols).fetchall())

ALLOWED_TABLES = {"trades", "ledger", "pools"}
def count_rows(conn, table):
    if table not in ALLOWED_TABLES:                           # identifiers can't be ? so we allow-list them
        raise ValueError(f"unknown table {table!r}")
    return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

print(count_rows(db, "trades"))

## 5. Schema design: let the database enforce your rules

Pandas happily holds a negative price or a duplicated row. A database can **refuse**. Constraints are guard rails that turn silent data corruption into immediate, loud errors at the exact moment bad data tries to enter, which is the cheapest time to catch it.

| Constraint | Meaning | Blockchain example |
|---|---|---|
| `PRIMARY KEY` | unique + not null identifier | `transactions.hash` |
| composite `PRIMARY KEY (a, b)` | the *pair* is unique | `prices_daily (coin_id, date)`: one price per coin per day |
| `NOT NULL` | value required | `from_addr` |
| `UNIQUE` | no duplicates | an address list |
| `CHECK (expr)` | arbitrary rule | `price_usd >= 0`, `is_error IN (0,1)` |
| `FOREIGN KEY` | must match a row elsewhere | `protocol_tvl.slug` → `protocols.slug` |

Here's the project schema from `db_tools.py`. Read it as a design, not just as syntax:

In [ ]:
print(SCHEMA[:1500], "\n  ...")

**Design decisions worth noticing:**

- **Natural keys.** `prices_daily` is keyed by `(coin_id, date)` because that *is* the identity of the data. Reloading the same day can't create a duplicate; it conflicts, and we upsert.
- **Dimension vs fact** (the same idea as Week 8's merge): `protocols` is a small **dimension** table (descriptions, one row per protocol); `protocol_tvl` is a big **fact** table (one row per protocol per day) pointing at it.
- **`value_wei TEXT`**: section 3.
- **An index** on `transactions(from_addr, ts)` for the most common question ("what did this wallet do, and when?"): section 6.

Constraints in action. Foreign keys are **OFF by default in SQLite** (a legacy quirk), so `connect()` in `db_tools.py` runs `PRAGMA foreign_keys = ON` every time:

In [ ]:
wh = init_db(connect())             # fresh in-memory warehouse using the project schema

def attempt(label, sql, params=()):
    try:
        with wh:
            wh.execute(sql, params)
        print(f"✅ {label}")
    except sqlite3.IntegrityError as e:
        print(f"🚫 {label}\n     refused: {e}")

attempt("negative price",       "INSERT INTO prices_daily (coin_id, date, price_usd) VALUES ('ethereum','2026-10-01', -5)")
attempt("good price",           "INSERT INTO prices_daily (coin_id, date, price_usd) VALUES ('ethereum','2026-10-01', 3250)")
attempt("duplicate day",        "INSERT INTO prices_daily (coin_id, date, price_usd) VALUES ('ethereum','2026-10-01', 3300)")
attempt("orphan TVL row (no such protocol)", "INSERT INTO protocol_tvl (slug, date, tvl_usd) VALUES ('ghost','2026-10-01', 1)")
attempt("bad flag",             "INSERT INTO transactions (hash, block_number, ts, from_addr, value_wei, is_error) VALUES ('0x1',1,1,'0xa','1',7)")

## 6. Indexes: why some queries take milliseconds and others minutes

Without an index, finding "all transactions from wallet X" means reading **every row** (a *scan*). An **index** is a sorted lookup structure (like a book's index) that jumps straight to the matches (a *search*). SQL doesn't change; the speed does.

Let's measure it on 200,000 synthetic transactions. `EXPLAIN QUERY PLAN` shows what SQLite intends to do, and it's the first thing to run on any slow query:

In [ ]:
rng = np.random.default_rng(1)
wallets = [f"0x{i:040x}" for i in range(5_000)]
rows = list(zip(range(200_000),
                rng.choice(wallets, 200_000).tolist(),
                rng.integers(1_700_000_000, 1_790_000_000, 200_000).tolist(),
                rng.random(200_000).tolist()))

big = sqlite3.connect(":memory:")
big.execute("CREATE TABLE tx (id INTEGER PRIMARY KEY, from_addr TEXT, ts INTEGER, value REAL)")
with big:
    big.executemany("INSERT INTO tx VALUES (?, ?, ?, ?)", rows)

q = "SELECT COUNT(*), SUM(value) FROM tx WHERE from_addr = ?"
target = wallets[123]

def timed(n=50):
    t0 = time.perf_counter()
    for _ in range(n):
        big.execute(q, (target,)).fetchone()
    return (time.perf_counter() - t0) / n * 1000

print("plan before index:", big.execute("EXPLAIN QUERY PLAN " + q, (target,)).fetchall()[0][3])
before = timed()

big.execute("CREATE INDEX idx_from ON tx(from_addr)")
print("plan after index :", big.execute("EXPLAIN QUERY PLAN " + q, (target,)).fetchall()[0][3])
after = timed()
print(f"\nper query: {before:.2f} ms -> {after:.3f} ms  ({before/after:,.0f}x faster)")

`SCAN tx` became `SEARCH tx USING INDEX`. The gap grows with table size: at 200k rows it's already large; at 200M it's the difference between instant and unusable.

**Index rules of thumb**
- Index the columns you **filter** (`WHERE`) and **join** (`ON`) on
- A composite index `(from_addr, ts)` serves `WHERE from_addr = ?` *and* `WHERE from_addr = ? AND ts > ?`, but not a filter on `ts` alone (**leftmost-prefix rule**)
- Primary keys are indexed automatically
- Indexes cost **disk space and slow writes**, so don't index everything; index what you actually query
- Always check with `EXPLAIN QUERY PLAN` instead of guessing

## 7. Loading data: fast, safe, repeatable

### 7.1 `executemany` inside one transaction

Each commit forces SQLite to flush to disk. Committing after every row is brutally slow; committing once is fast. Measured on a real file:

In [ ]:
data = [(i, f"0x{i:040x}", float(i)) for i in range(3_000)]

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "speed.db")

    c1 = sqlite3.connect(path)
    c1.execute("CREATE TABLE a (id INTEGER PRIMARY KEY, addr TEXT, v REAL)")
    t0 = time.perf_counter()
    for row in data:                                  # commit per row
        c1.execute("INSERT INTO a VALUES (?, ?, ?)", row)
        c1.commit()
    slow = time.perf_counter() - t0

    c1.execute("CREATE TABLE b (id INTEGER PRIMARY KEY, addr TEXT, v REAL)")
    t0 = time.perf_counter()
    with c1:                                          # one transaction, one flush
        c1.executemany("INSERT INTO b VALUES (?, ?, ?)", data)
    fast = time.perf_counter() - t0
    c1.close()

print(f"row-by-row commits: {slow*1000:7.0f} ms")
print(f"executemany once  : {fast*1000:7.0f} ms   ({slow/fast:,.0f}x faster)")

### 7.2 Making loads re-runnable: `INSERT OR IGNORE` and UPSERT

A pipeline you can't safely run twice is a pipeline that will bite you at 3 a.m. Two tools:

| | Behaviour on conflict | Use for |
|---|---|---|
| `INSERT OR IGNORE` | silently skip the new row | **Immutable** facts: a mined transaction never changes |
| `INSERT ... ON CONFLICT (key) DO UPDATE SET col = excluded.col` | **update** the existing row | **Revisable** facts: today's price changes during the day |

`excluded` is SQLite's name for "the row I just tried to insert".

(`INSERT OR REPLACE` also exists, but it **deletes then reinserts**, which can trigger cascading deletes and resets columns you didn't mention. Prefer upsert.)

In [ ]:
db.execute("CREATE TABLE px (coin TEXT, date TEXT, price REAL, PRIMARY KEY (coin, date))")

UPSERT = '''INSERT INTO px (coin, date, price) VALUES (?, ?, ?)
            ON CONFLICT(coin, date) DO UPDATE SET price = excluded.price'''

with db: db.execute(UPSERT, ("ethereum", "2026-10-01", 3250.0))      # first load
with db: db.execute(UPSERT, ("ethereum", "2026-10-01", 3262.5))      # re-run later with a corrected price
with db: db.execute(UPSERT, ("ethereum", "2026-10-01", 3262.5))      # run it AGAIN: nothing changes

print(db.execute("SELECT * FROM px").fetchall())                     # still ONE row, latest value

This is **idempotency**: running an operation N times has the same effect as running it once. It's the most valuable property a data pipeline can have, and it's why every table in `db_tools.py` has a natural primary key.

### 7.3 `DataFrame.to_sql`, and its trap

`df.to_sql("table", conn, if_exists=..., index=False)` is a one-line loader. It's great for quick exploration but **not** for a warehouse:

| `if_exists` | Does | Problem |
|---|---|---|
| `"fail"` (default) | error if the table exists | |
| `"replace"` | **drops and recreates the table** | **destroys your constraints, keys and indexes** |
| `"append"` | inserts rows | duplicates on re-run; no upsert |

In [ ]:
demo = sqlite3.connect(":memory:")
demo.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, price REAL CHECK (price > 0))")
print("before :", [(r[1], r[2], 'PK' if r[5] else '') for r in demo.execute("PRAGMA table_info(t)")])

pd.DataFrame({"id": [1, 2], "price": [10.0, 20.0]}).to_sql("t", demo, if_exists="replace", index=False)
print("after  :", [(r[1], r[2], 'PK' if r[5] else '') for r in demo.execute("PRAGMA table_info(t)")], " <- primary key and CHECK are gone")

**Pattern for the warehouse:** create the schema yourself (constraints and all), then load with `executemany` + upsert (what `db_tools.upsert_*` do). Use `to_sql` only for scratch tables.

*Also*: `to_sql(..., index=False)` unless you want the DataFrame's 0..n row labels written as a column.

## 8. Populating the warehouse

Time to fill the project database. `run_pipeline` (built in `lesson_advanced.ipynb`; used as a black box here) pulls from the three clients (offline fixtures) and upserts into the schema from section 5.

In [ ]:
wh = init_db(connect())
summary = run_pipeline(wh, FixtureSession(), as_of="2026-10-01")
print("rows loaded this run:", summary)
print("table counts        :", table_counts(wh))

## 9. Querying: your SQL, plus what's different

You know `SELECT`, `WHERE`, `GROUP BY`. Here are the patterns that matter on *this* data, in SQLite dialect. A tiny helper prints any query as a DataFrame so results look nice:

In [ ]:
def q(sql, params=()):
    '''Run SQL, return a DataFrame (read_sql_query is covered properly in section 10).'''
    return pd.read_sql_query(sql, wh, params=params)

q('''SELECT coin_id, COUNT(*) AS days, MIN(date) AS first_day, MAX(date) AS last_day,
           ROUND(AVG(price_usd), 2) AS avg_price
    FROM prices_daily GROUP BY coin_id''')

### 9.1 `GROUP BY` ... `HAVING`

`HAVING` filters **groups** after aggregation, while `WHERE` filters **rows** before. (Same as DuneSQL.)

In [ ]:
q('''SELECT category, COUNT(*) AS protocols, ROUND(SUM(tvl_usd) / 1e9, 1) AS tvl_bn
    FROM protocols
    GROUP BY category
    HAVING SUM(tvl_usd) > 5e9
    ORDER BY tvl_bn DESC''')

### 9.2 `JOIN`: fact table to dimension

Latest TVL per protocol, with names and categories from the dimension table. Note the **scalar subquery** `(SELECT MAX(date) ...)` to pin "the latest day".

In [ ]:
q('''SELECT p.name, p.category, t.date, ROUND(t.tvl_usd / 1e9, 2) AS tvl_bn
    FROM protocol_tvl t
    JOIN protocols p ON p.slug = t.slug
    WHERE t.date = (SELECT MAX(date) FROM protocol_tvl)
    ORDER BY t.tvl_usd DESC''')

`RIGHT JOIN` and `FULL OUTER JOIN` exist only in SQLite **3.39+** (2022). On older versions, flip the table order and use `LEFT JOIN`.

### 9.3 CTEs (`WITH`): name your steps

A **Common Table Expression** is a named subquery. It's how you keep a multi-step query readable, exactly as you would in DuneSQL. Here, 30-day TVL change per protocol:

In [ ]:
q('''WITH latest AS (
        SELECT slug, tvl_usd FROM protocol_tvl
        WHERE date = (SELECT MAX(date) FROM protocol_tvl)
    ),
    prior AS (
        SELECT slug, tvl_usd FROM protocol_tvl
        WHERE date = date((SELECT MAX(date) FROM protocol_tvl), '-30 days')
    )
    SELECT l.slug,
           ROUND((l.tvl_usd / p.tvl_usd - 1) * 100, 2) AS change_30d_pct
    FROM latest l JOIN prior p USING (slug)
    ORDER BY change_30d_pct DESC''')

### 9.4 Window functions

Everything from Week 8's "`shift`, `rolling`, `rank`" table has a SQL form here, and **SQLite supports them since 3.25**. Daily return with `LAG`, a 7-day moving average with a frame, and a named window to avoid repeating yourself:

In [ ]:
q('''SELECT date,
           price_usd,
           ROUND((price_usd / LAG(price_usd) OVER w - 1) * 100, 3)                         AS return_pct,
           ROUND(AVG(price_usd) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS ma7
    FROM prices_daily
    WHERE coin_id = 'ethereum'
    WINDOW w AS (ORDER BY date)
    ORDER BY date DESC LIMIT 5''')

**Top-N per group** with `ROW_NUMBER()`: the largest protocol in each category (Week 8's `groupby().idxmax()` / `rank`):

In [ ]:
q('''SELECT category, name, ROUND(tvl_usd / 1e9, 1) AS tvl_bn
    FROM (SELECT category, name, tvl_usd,
                 ROW_NUMBER() OVER (PARTITION BY category ORDER BY tvl_usd DESC) AS rn
          FROM protocols)
    WHERE rn = 1
    ORDER BY tvl_usd DESC''')

### 9.5 Dates and times

Timestamps are stored as integers, so conversion happens in the query. The functions to know:

| Task | SQLite | DuneSQL |
|---|---|---|
| unix seconds → date | `date(ts, 'unixepoch')` | `date(from_unixtime(ts))` |
| truncate to day | `date(ts, 'unixepoch')` | `date_trunc('day', t)` |
| truncate to month | `date(d, 'start of month')` | `date_trunc('month', d)` |
| add/subtract | `date(d, '-7 days')` | `d - interval '7' day` |
| day of week (0=Sun) | `strftime('%w', d)` | `day_of_week(d)` |
| hour | `strftime('%H', ts, 'unixepoch')` | `hour(t)` |
| days between | `julianday(b) - julianday(a)` | `date_diff('day', a, b)` |

Daily activity of our demo wallet: transaction count, ETH moved (converted from text-wei with a float cast, fine for display), failure rate, and gas spent:

In [ ]:
q('''SELECT date(ts, 'unixepoch')                                       AS day,
           COUNT(*)                                                    AS txs,
           ROUND(SUM(CAST(value_wei AS REAL)) / 1e18, 2)               AS eth_moved,
           ROUND(100.0 * SUM(is_error) / COUNT(*), 1)                  AS fail_pct,
           ROUND(SUM(gas_used * CAST(gas_price_wei AS REAL)) / 1e18, 4) AS gas_eth
    FROM transactions
    GROUP BY day ORDER BY day LIMIT 6''')

(For display and rough analysis, `CAST(... AS REAL)` is fine. For *accounting* that must reconcile to the wei, sum in Python ints, as section 3 showed.)

### 9.6 Join on-chain to off-chain: gas spend in USD

The query that justifies the whole week: transactions from one source, prices from another, joined **inside SQL** on the date.

In [ ]:
q('''SELECT date(t.ts, 'unixepoch') AS day,
           COUNT(*) AS txs,
           ROUND(SUM(t.gas_used * CAST(t.gas_price_wei AS REAL) / 1e18 * p.price_usd), 2) AS gas_usd,
           ROUND(SUM(CAST(t.value_wei AS REAL) / 1e18 * p.price_usd) / 1e3, 1)          AS moved_usd_k
    FROM transactions t
    JOIN prices_daily p ON p.coin_id = 'ethereum' AND p.date = date(t.ts, 'unixepoch')
    GROUP BY day ORDER BY gas_usd DESC LIMIT 5''')

## 10. The Pandas ↔ SQL bridge

`pd.read_sql_query(sql, conn, params=..., parse_dates=...)` runs SQL and returns a DataFrame. Two arguments matter:

- **`params`**: always use it (placeholders, section 4), never f-strings
- **`parse_dates`**: SQLite has no date type, so dates arrive as strings; this converts them to real `datetime64` so `.dt` and `resample` work

In [ ]:
eth = pd.read_sql_query(
    "SELECT date, price_usd FROM prices_daily WHERE coin_id = ? ORDER BY date",
    wh, params=("ethereum",), parse_dates=["date"])
print(eth.dtypes.to_dict())
eth.tail(3)

### 10.1 Same question, two languages: do they agree?

The 7-day moving average computed in SQL (window function) and in Pandas (`rolling`) **should match**, and checking that is how you build trust in both. One genuine difference to notice: SQL's frame `ROWS BETWEEN 6 PRECEDING` averages *whatever rows exist*, so the first six values use partial windows, while Pandas' `rolling(7)` returns `NaN` until it has 7.

In [ ]:
sql_ma = pd.read_sql_query('''SELECT date,
          AVG(price_usd) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
          FROM prices_daily WHERE coin_id = 'ethereum' ORDER BY date''', wh)["ma7"]
pd_ma = eth["price_usd"].rolling(7).mean()

print("first 6 rows  -> SQL:", sql_ma.head(6).round(1).tolist()[:3], "... | Pandas:", pd_ma.head(6).tolist()[:3], "...")
print("from row 7 on -> identical:", np.allclose(sql_ma[6:], pd_ma[6:]))

### 10.2 Which tool for which job?

| Task | Reach for | Why |
|---|---|---|
| Filter/aggregate a big table, pull the answer | **SQL** | Done in the database; only the result crosses into Python |
| Join tables that already live in the DB | **SQL** | Indexes + one round trip |
| Reshape (pivot), plot, model, custom Python logic | **Pandas** | Ergonomic, rich ecosystem |
| Exact big-integer maths | **Python ints** | Neither SQLite nor floats are exact |
| Anything iterative or API-driven | **Python** | SQL can't call an API |

**The workflow:** `SQL` to cut a huge table down → `read_sql_query` → `Pandas` to analyse and plot. That's the Phase 2 capstone in one sentence.

## 11. DuneSQL (Trino) vs SQLite cheat sheet

You'll write SQL in both worlds; these are the differences that actually bite:

| Topic | DuneSQL / Trino | SQLite |
|---|---|---|
| Date truncation | `date_trunc('day', t)` | `date(t)` / `strftime(...)` |
| Intervals | `t - interval '7' day` | `date(t, '-7 days')` |
| Date diff | `date_diff('day', a, b)` | `julianday(b) - julianday(a)` |
| Addresses | `varbinary` (`0xabc...`), compare to `0x...` literals | **lowercase hex `TEXT`**: normalise case yourself (`normalize_address`) |
| Big numbers | `uint256`, `DECIMAL(38,0)` | **int64 only**: use TEXT + Python ints |
| Case-insensitive match | `ILIKE` / `lower()` | `LIKE` is already case-insensitive for ASCII |
| Percentiles | `approx_percentile(x, 0.9)` | none built in: use Pandas `quantile` or `NTILE` |
| Arrays / `UNNEST` | native | none: normalise into another table |
| `RIGHT` / `FULL JOIN` | yes | yes since 3.39 |
| `FILTER (WHERE ...)` on aggregates | yes | yes since 3.30 |
| Booleans | `true`/`false` | `1`/`0` |
| Integer division | `/` on ints may truncate | `/` on ints truncates: use `1.0 *` or `CAST` |
| Types enforced? | strictly | **loosely**, so add `CHECK`s |

The last two matter in practice. `7 / 2` is `3` in SQLite; write `7 / 2.0`.

## 12. Inspecting and maintaining your database

The database describes itself. Four things worth knowing:

In [ ]:
print("tables :", [r[0] for r in wh.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])
print("indexes:", [r[0] for r in wh.execute("SELECT name FROM sqlite_master WHERE type='index' AND name NOT LIKE 'sqlite_%'")])
print()
for r in wh.execute("PRAGMA table_info(prices_daily)"):                    # cid, name, type, notnull, default, pk
    print(f"  {r['name']:<11} {r['type']:<5} notnull={r['notnull']} pk={r['pk']}")
print()
print("foreign-key violations:", wh.execute("PRAGMA foreign_key_check").fetchall(), "(empty = healthy)")

**Saving to a file** (what `pipeline.py` does): `connect("chainlens.db")` instead of `":memory:"`. That one file *is* your warehouse; copy it, back it up, open it elsewhere.

```python
# safe live backup (works while the database is in use)
src = connect("chainlens.db")
dst = sqlite3.connect("backup.db")
src.backup(dst)
```

```bash
# the sqlite3 command line (installed on most systems; or use "DB Browser for SQLite" for a GUI)
sqlite3 chainlens.db
sqlite> .tables
sqlite> .schema prices_daily
sqlite> .headers on
sqlite> .mode column
sqlite> SELECT * FROM etl_runs;
```

`VACUUM` rebuilds the file to reclaim space after large deletes. Add `*.db` to `.gitignore`: **a database is generated output, not source code**. Commit the code that creates it.

## 13. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| f-string SQL | Wrong rows, errors on quotes, injection | `?` placeholders |
| Forgetting `PRAGMA foreign_keys = ON` | Orphan rows accepted | Use `db_tools.connect()` |
| No transaction around a load | Half-loaded data after a crash | `with conn:` |
| Commit per row | Loads take minutes | `executemany` in one transaction |
| `to_sql(if_exists="replace")` | Constraints and keys vanish | Create schema yourself; upsert |
| Wei in INTEGER/REAL | Silent precision loss or int64 overflow | TEXT + Python ints |
| Mixed-case addresses | Same wallet looks like two | Store lowercase |
| Dates as free-form strings | Wrong sorting, failed joins | Strict `'YYYY-MM-DD'` |
| No index on a filtered column | Queries slow as data grows | `EXPLAIN QUERY PLAN`, then index |
| Integer division | `7 / 2 = 3` | `7 / 2.0` |
| Relying on row order without `ORDER BY` | Results vary | Always `ORDER BY` |

## Lesson summary

| Concept | Detail |
|---|---|
| SQLite | A SQL database in one file, built into Python (`sqlite3`) |
| Connection / cursor / Row | `connect()`, `execute()`, `row_factory = sqlite3.Row` |
| Transactions | `with conn:` = commit on success, rollback on error |
| Types | Dynamic; no date/bool/uint256; **wei as TEXT** |
| Parameters | `?` / `:name`; identifiers via allow-list |
| Constraints | `PRIMARY KEY`, `NOT NULL`, `CHECK`, `FOREIGN KEY` (**pragma on**) |
| Indexes | Turn scans into searches; verify with `EXPLAIN QUERY PLAN` |
| Loading | `executemany` + one transaction; upsert for idempotency |
| `to_sql` | Quick and dirty; `replace` destroys the schema |
| SQL features | CTEs, window functions (`LAG`, `ROW_NUMBER`, frames), date functions |
| Pandas bridge | `read_sql_query(sql, conn, params=, parse_dates=)` |
| Division of labour | SQL cuts the data down; Pandas analyses and plots |

> 📖 **Go deeper** (curated list and study order in `RESOURCES.md`): the official Python [`sqlite3` docs](https://docs.python.org/3/library/sqlite3.html), SQLite's own [documentation](https://www.sqlite.org/docs.html), and *Python for Everybody* Chapter 15 *Using Databases and SQL* (free at `pythonlearn.com`).

## What's next

**`lesson_advanced.ipynb`**: connect Part 1 and Part 2 into one **re-runnable ETL pipeline**: incremental loads, logging, data-quality checks, a CLI, and the analytics that come out the other end. It's the engine of the Phase 2 capstone.

## Commit your work

```bash
cd ~/Desktop/python-for-blockchain-analytics
git add phase-2-python-for-data/week-10-apis-sql/
git commit -m "phase-2/week-10: SQLite and SQL from Python"
git push
```

Make sure `*.db` and `.env` are in your `.gitignore` first (`git status` should not list a database file).